# Essayons des exemples réservés

Observer une nouvelle passe sans continuer à ajuster le modèle.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#entrainer-ou-evaluer).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Les trois exemples appris sont bien classés. Ce résultat est encourageant pour vérifier le programme, mais ces lignes ont guidé chacune de ses mises à jour. Regardons maintenant quatre observations qui n’ont participé à aucun backward.

| Extrait fictif | RMS choisie | Facteur de crête choisi | Annotation |
| --- | --- | --- | --- |
| Voix D | 0,12 | 1,8 | 0 |
| Percussion E | 0,18 | 4,0 | 1 |
| Voix F | 0,16 | 2,2 | 0 |
| Percussion G | 0,22 | 3,8 | 1 |

Ces quatre lignes sont aussi choisies à la main. Elles servent à comprendre le protocole de validation, pas à estimer une performance réelle sur des sons.

Le modèle reste celui de la première boucle, après ses 201 pas. Nous lui donnerons les mesures réservées, puis comparerons sa réponse aux annotations. Ses poids doivent rester inchangés pendant cette observation.

Deux réglages accompagnent habituellement cette validation. modele.eval() sélectionne le comportement d’évaluation des couches. torch.no_grad() désactive le suivi des nouveaux calculs pour une dérivation. Ils répondent à deux besoins différents.

Nous faisons bien une nouvelle passe : ni les anciens scores ni l’ancienne perte ne décrivent ces nouvelles entrées. Le critère mesure leur désaccord avec leurs propres annotations. Aucun backward ni step n’apparaît dans cette partie.

Prenez le temps de distinguer les garanties. eval ne gèle pas les paramètres et ne coupe pas autograd. no_grad ne sélectionne pas le comportement des couches. L’absence de mise à jour vient ici de notre protocole : nous n’appliquons aucun step.

Notre Linear donne la même règle de calcul dans les deux modes. Le réglage eval prépare aussi le cas de couches sensibles au mode : Dropout cesse son masquage aléatoire, par exemple. Nous n’avons pas besoin d’en ajouter une pour observer les quatre entrées réservées.

Le test == garde un booléen par exemple. sum compte les True et item extrait ce compte dans Python. Nous pouvons ainsi lire les décisions et leur accord, tout en gardant la perte comme mesure de leurs scores.

| Mode des couches | Suivi des nouveaux calculs | Conséquence |
| --- | --- | --- |
| train | Activé | Réglage ordinaire d’apprentissage, avant backward et step. |
| eval | Désactivé par no_grad | Réglage ordinaire de validation présenté ici. |
| eval | Activé | Le comportement d’évaluation peut encore être dérivé. |
| train | Désactivé par no_grad | Des couches restent en comportement d’entraînement, sans nouveau graphe. |

Ce tableau est un rappel, pas quatre routines à apprendre avant de pouvoir valider.

Les quatre décisions concordent avec les annotations, mais le coût vaut encore environ 0,310 : les scores ne donnent pas une probabilité de un aux bonnes classes. Quatre accords sur quatre restent une vérification minuscule, sur des nombres construits pour le cours.

Une vraie évaluation demande des extraits représentatifs et une séparation adaptée aux enregistrements. La validation guide nos choix ; la consulter de façon répétée peut aussi nous adapter à ses particularités. Un test indépendant servira à l’évaluation finale.

Le trajet complet est maintenant visible : mesures, scores, perte, gradients, mise à jour, puis observation séparée. La prochaine difficulté sera plus pratique : parcourir davantage d’exemples sans donner tout le corpus au modèle d’un seul coup.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Le début autonome reprend exactement la préparation puis les 201 pas de la première boucle. Les quatre observations réservées ne sont pas encore utilisées.

In [4]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Définir la règle avant de l’utiliser

Cette cellule définit un objet ou une fonction. Son corps ne sera utilisé que lors d’un appel. Nous gardons la définition entière pour ne pas casser la syntaxe.

In [6]:
class Classifieur(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)
    def forward(self, mesures):
        return self.linear(mesures)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
modele = Classifieur()


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
with torch.no_grad():
    modele.linear.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.linear.bias.zero_()


### Choisir les paramètres et leur mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [9]:
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)


### Répéter la même étape sur les données

Le corps indenté reste dans une seule cellule. Les opérations se répètent dans l’ordre écrit ; les sorties permettront de suivre leurs effets.

In [10]:
for epoque in range(201):
    optimiseur.zero_grad()
    scores = modele(X)
    perte = critere(scores, cibles)
    perte.backward()
    optimiseur.step()


### Calculer l’état suivant

Ces deux tableaux sont indépendants de X et cibles, avec les mêmes conventions de colonnes et de classes.

In [11]:
X_reserve = torch.tensor([[0.12, 1.8], [0.18, 4.0], [0.16, 2.2], [0.22, 3.8]])
cibles_reserve = torch.tensor([0, 1, 0, 1], dtype=torch.long)


In [12]:
observer('X_reserve', X_reserve)
observer('cibles_reserve', cibles_reserve)

X_reserve forme (4, 2) type torch.float32 requires_grad False
  valeurs : [[0.11999999731779099, 1.7999999523162842], [0.18000000715255737, 4.0], [0.1599999964237213, 2.200000047683716], [0.2199999988079071, 3.799999952316284]]
cibles_reserve forme (4,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0, 1]


### Calculer l’état suivant

tolist produit des listes Python des valeurs présentes. Elles permettront de vérifier que poids et biais n’ont pas changé.

In [13]:
poids_avant = modele.linear.weight.tolist()
biais_avant = modele.linear.bias.tolist()


In [14]:
observer('poids_avant', poids_avant)
observer('biais_avant', biais_avant)

poids_avant : 2 éléments ; formes des tenseurs : []
biais_avant = [1.8992172479629517, -1.8992173671722412]


### Calculer l’état suivant

eval sélectionne le comportement des couches ; no_grad encadre les nouveaux scores, la perte et les décisions. Aucun step n’est appelé.

In [15]:
modele.eval()


Classifieur(
  (linear): Linear(in_features=2, out_features=2, bias=True)
)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
with torch.no_grad():
    scores_reserve = modele(X_reserve)
    perte_reserve = critere(scores_reserve, cibles_reserve)
    predictions_reserve = scores_reserve.argmax(dim=1)


### Calculer l’état suivant

La comparaison conserve l’ordre des quatre extraits. La somme des booléens compte les décisions conformes aux annotations.

In [17]:
accords_reserve = predictions_reserve == cibles_reserve
nombre_correct = accords_reserve.sum().item()


In [18]:
observer('accords_reserve', accords_reserve)
observer('nombre_correct', nombre_correct)

accords_reserve forme (4,) type torch.bool requires_grad False
  valeurs : [True, True, True, True]
nombre_correct = 4


### Calculer l’état suivant

Les nouvelles listes des paramètres sont égales aux instantanés. Le résultat décrit ce minuscule jeu choisi à la main.

In [19]:
poids_identiques = modele.linear.weight.tolist() == poids_avant
biais_identiques = modele.linear.bias.tolist() == biais_avant


In [20]:
observer('poids_identiques', poids_identiques)
observer('biais_identiques', biais_identiques)

poids_identiques = True
biais_identiques = True


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [21]:
print("decisions reservees :", predictions_reserve.tolist())
print("cibles reservees :", cibles_reserve.tolist())
print("accords :", nombre_correct, "/", len(cibles_reserve))
print("perte reservee :", round(perte_reserve.item(), 3))


decisions reservees : [0, 1, 0, 1]
cibles reservees : [0, 1, 0, 1]
accords : 4 / 4
perte reservee : 0.31


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [22]:
print("mode entrainement :", modele.training)
print("graphe des nouveaux scores :", scores_reserve.requires_grad)
print("poids et biais inchanges :", poids_identiques, biais_identiques)

mode entrainement : False
graphe des nouveaux scores : False
poids et biais inchanges : True True


In [23]:
assert bool(tuple(X_reserve.shape) == (4, 2) and cibles_reserve.tolist() == [0, 1, 0, 1]), 'entrainer-ou-evaluer/main: tuple(X_reserve.shape) == (4, 2) and cibles_reserve.tolist() == [0, 1, 0, 1]'
assert bool(predictions_reserve.tolist() == [0, 1, 0, 1] and nombre_correct == 4), 'entrainer-ou-evaluer/main: predictions_reserve.tolist() == [0, 1, 0, 1] and nombre_correct == 4'
assert bool(poids_identiques and biais_identiques and not modele.training), 'entrainer-ou-evaluer/main: poids_identiques and biais_identiques and not modele.training'
assert bool(not scores_reserve.requires_grad and not perte_reserve.requires_grad), 'entrainer-ou-evaluer/main: not scores_reserve.requires_grad and not perte_reserve.requires_grad'
assert bool(abs(perte_reserve.item() - 0.30957514) < 2e-6), 'entrainer-ou-evaluer/main: abs(perte_reserve.item() - 0.30957514) < 2e-6'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un modèle est en eval, mais sans no_grad. Sa nouvelle sortie possède-t-elle requires_grad=True ?

In [24]:
exo_modele = torch.nn.Linear(1,1).eval()
exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = ...  # Remplacer les points par votre réponse.
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

eval change le comportement des modules concernés ; il ne désactive pas autograd.

</details>

### Exercice 2

Dans no_grad, cette nouvelle opération conserve-t-elle un graphe de dérivées ?

In [25]:
exo_modele = torch.nn.Linear(1,1).eval()
with torch.no_grad():
    exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = ...  # Remplacer les points par votre réponse.
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Nous regardons la sortie d’une opération nouvelle, pas un tenseur créé avant le contexte.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_modele = torch.nn.Linear(1,1).eval()
exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = True
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_modele = torch.nn.Linear(1,1).eval()
with torch.no_grad():
    exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = False
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [28]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()